In [ ]:
# ---------------------------------------------
# IRIS SPECIES PREDICTION USING PYTHON
# ---------------------------------------------

# Import required libraries
import pandas as pd
import numpy as np

# ------------------------------------------------
# STEP 1 : LOAD DATASET
# ------------------------------------------------

# Load Iris dataset
# header=None because dataset has no column names
df = pd.read_csv("Iris.csv", header=None)

# Assign column names
df.columns = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
    "Species"
]

# Display first 5 rows
display(df.head())

# Dataset information
print(df.info())


# ------------------------------------------------
# STEP 2 : DATA CLEANING
# ------------------------------------------------

# Check missing values
print(df.isnull().sum())

# Check ? values
print((df == "?").sum())

# Remove missing values
df.dropna(inplace=True)

# Remove negative values
df = df[
    (df["SepalLengthCm"] >= 0) &
    (df["SepalWidthCm"] >= 0) &
    (df["PetalLengthCm"] >= 0) &
    (df["PetalWidthCm"] >= 0)
]

# Verify cleaned dataset
print(df.info())

display(df.head())


# ------------------------------------------------
# STEP 3 : OUTLIER DETECTION AND REMOVAL
# ------------------------------------------------

# Select numerical columns
numeric_cols = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]

# Calculate Q1 and Q3
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

print(IQR)

# Remove outliers using IQR method
df_no_outliers = df[
    ~((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
      (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)
]

# Display cleaned dataset
display(df_no_outliers.head())

print("Original Shape:", df.shape)
print("New Shape:", df_no_outliers.shape)


# ------------------------------------------------
# STEP 4 : DATA TRANSFORMATION
# ------------------------------------------------

# Separate features and target
X = df_no_outliers.drop("Species", axis=1)

y = df_no_outliers["Species"]

# Normalize data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Split dataset into training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)


# ------------------------------------------------
# STEP 5 : LOGISTIC REGRESSION MODEL
# ------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create model
lr_model = LogisticRegression()

# Train model
lr_model.fit(X_train, y_train)

# Prediction
lr_pred = lr_model.predict(X_test)

# Accuracy
lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_accuracy)


# ------------------------------------------------
# STEP 6 : NAIVE BAYES MODEL
# ------------------------------------------------

from sklearn.naive_bayes import GaussianNB

# Create model
nb_model = GaussianNB()

# Train model
nb_model.fit(X_train, y_train)

# Prediction
nb_pred = nb_model.predict(X_test)

# Accuracy
nb_accuracy = accuracy_score(y_test, nb_pred)

print("Naive Bayes Accuracy:", nb_accuracy)


# ------------------------------------------------
# STEP 7 : COMPARE ACCURACY
# ------------------------------------------------

if lr_accuracy > nb_accuracy:
    print("Logistic Regression performs better")

elif nb_accuracy > lr_accuracy:
    print("Naive Bayes performs better")

else:
    print("Both models perform equally")